In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import socket
import json
import os
from datetime import datetime

HOST = '127.0.0.1'
PORT = 65432

class CinemaClientApp:
    def __init__(self, master):
        self.master = master
        master.title("NewLine Cinema Ticket System")
        master.geometry("800x700")
        master.resizable(True, True)

        self.client_socket = None
        self.movies = []
        self.selected_movie_id = None

        self.create_widgets()
        self.connect_to_server()
        self.load_movies()

    def connect_to_server(self):
        try:
            self.client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            self.client_socket.connect((HOST, PORT))
            self.display_message("Connected to server.", is_error=False)
        except ConnectionRefusedError:
            self.display_message("Connection to server failed. Ensure server is running.", is_error=True)
            self.master.destroy()
        except Exception as e:
            self.display_message(f"An error occurred connecting: {e}", is_error=True)
            self.master.destroy()

    def send_request(self, action, data={}):
        if not self.client_socket:
            self.display_message("Not connected to server.", is_error=True)
            return {"status": "error", "message": "Not connected to server."}
        try:
            request = {"action": action, "data": data}
            self.client_socket.sendall(json.dumps(request).encode('utf-8'))
            response_data = self.client_socket.recv(4096)
            response = json.loads(response_data.decode('utf-8'))
            return response
        except json.JSONDecodeError:
            self.display_message("Invalid JSON response from server.", is_error=True)
            return {"status": "error", "message": "Invalid JSON response from server."}
        except socket.error as e:
            self.display_message(f"Socket error during communication: {e}", is_error=True)
            return {"status": "error", "message": f"Socket communication error: {e}"}
        except Exception as e:
            self.display_message(f"An unexpected error occurred during request: {e}", is_error=True)
            return {"status": "error", "message": f"Unexpected error: {e}"}

    def load_movies(self):
        response = self.send_request('get_movies')
        if response['status'] == 'success':
            self.movies = response['data']
            movie_titles = [movie['title'] for movie in self.movies]
            self.movie_selection_combobox['values'] = movie_titles
            if self.movies:
                self.movie_selection_combobox.set(self.movies[0]['title'])
                self.on_movie_selected(None)
            else:
                self.movie_selection_combobox.set("")
                self.display_movie_details({})
            self.display_message("Movies loaded successfully.", is_error=False)
        else:
            self.display_message(f"Failed to load movies: {response['message']}", is_error=True)

    def on_movie_selected(self, event):
        selected_title = self.movie_selection_combobox.get()
        selected_movie = next((movie for movie in self.movies if movie['title'] == selected_title), None)

        if selected_movie:
            self.selected_movie_id = selected_movie['id']
            self.display_movie_details(selected_movie)
        else:
            self.selected_movie_id = None
            self.display_movie_details({})
            self.display_message("Please select a movie.", is_error=True)

    def display_movie_details(self, movie_data):
        self.detail_title_var.set(movie_data.get('title', ''))
        self.detail_room_var.set(f"Room: {movie_data.get('cinema_room', '')}")
        self.detail_release_var.set(f"Release: {movie_data.get('release_date', '')}")
        self.detail_end_var.set(f"End: {movie_data.get('end_date', '')}")
        self.detail_tickets_var.set(f"Tickets Available: {movie_data.get('tickets_available', '')}")
        self.detail_price_var.set(f"Price: R{movie_data.get('ticket_price', ''):.2f}" if 'ticket_price' in movie_data else '')

        self.update_title_entry.delete(0, tk.END)
        self.update_title_entry.insert(0, movie_data.get('title', ''))
        self.update_room_entry.delete(0, tk.END)
        self.update_room_entry.insert(0, movie_data.get('cinema_room', ''))
        self.update_release_entry.delete(0, tk.END)
        self.update_release_entry.insert(0, movie_data.get('release_date', ''))
        self.update_end_entry.delete(0, tk.END)
        self.update_end_entry.insert(0, movie_data.get('end_date', ''))
        self.update_tickets_entry.delete(0, tk.END)
        self.update_tickets_entry.insert(0, movie_data.get('tickets_available', ''))
        self.update_price_entry.delete(0, tk.END)
        self.update_price_entry.insert(0, movie_data.get('ticket_price', ''))


    def purchase_tickets(self):
        if self.selected_movie_id is None:
            self.display_message("Please select a movie first.", is_error=True)
            return

        try:
            num_tickets = int(self.quantity_entry.get())
            customer_name = self.customer_name_entry.get().strip()

            if num_tickets <= 0:
                raise ValueError("Number of tickets must be positive.")
            if not customer_name:
                raise ValueError("Customer name cannot be empty.")

            response = self.send_request('record_sale', {
                'movie_id': self.selected_movie_id,
                'customer_name': customer_name,
                'number_of_tickets': num_tickets
            })

            if response['status'] == 'success':
                sale_info = response['data']
                self.total_cost_label.config(text=f"Total Cost: R{sale_info['total_cost']:.2f}")
                self.display_message(f"Tickets purchased successfully! Remaining: {sale_info['tickets_remaining']}", is_error=False)
                self.generate_receipt(sale_info, customer_name, num_tickets, sale_info['total_cost'])
                self.load_movies()
            else:
                self.display_message(f"Purchase failed: {response['message']}", is_error=True)
        except ValueError as ve:
            self.display_message(f"Invalid input: {ve}", is_error=True)
        except Exception as e:
            self.display_message(f"An unexpected error occurred during purchase: {e}", is_error=True)

    def generate_receipt(self, sale_info, customer_name, num_tickets, total_cost):
        receipt_filename = f"receipt_{sale_info['movie_title'].replace(' ', '_')}_{datetime.now().strftime('%Y%m%d%H%M%S')}.txt"
        receipt_content = f"""
        --- NewLine Cinema Receipt ---
        Movie Title: {sale_info['movie_title']}
        Movie ID: {sale_info['movie_id']}
        Customer Name: {customer_name}
        Number of Tickets: {num_tickets}
        Total Cost: R{total_cost:.2f}
        Date of Sale: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
        -----------------------------
        Thank you for your purchase!
        """
        try:
            with open(receipt_filename, 'w') as f:
                f.write(receipt_content)
            self.display_message(f"Receipt saved as {receipt_filename}", is_error=False)
        except IOError as e:
            self.display_message(f"Failed to save receipt: {e}", is_error=True)

    def add_movie(self):
        try:
            title = self.new_title_entry.get().strip()
            room = int(self.new_room_entry.get())
            release = self.new_release_entry.get().strip()
            end = self.new_end_entry.get().strip()
            tickets = int(self.new_tickets_entry.get())
            price = float(self.new_price_entry.get())

            if not all([title, release, end]):
                raise ValueError("All new movie fields must be filled.")

            response = self.send_request('add_movie', {
                'title': title,
                'cinema_room': room,
                'release_date': release,
                'end_date': end,
                'tickets_available': tickets,
                'ticket_price': price
            })

            if response['status'] == 'success':
                self.display_message(f"Movie '{title}' added successfully!", is_error=False)
                self.new_title_entry.delete(0, tk.END)
                self.new_room_entry.delete(0, tk.END)
                self.new_release_entry.delete(0, tk.END)
                self.new_end_entry.delete(0, tk.END)
                self.new_tickets_entry.delete(0, tk.END)
                self.new_price_entry.delete(0, tk.END)
                self.load_movies()
            else:
                self.display_message(f"Failed to add movie: {response['message']}", is_error=True)
        except ValueError as ve:
            self.display_message(f"Invalid input for new movie: {ve}. Check numbers and dates (YYYY-MM-DD).", is_error=True)
        except Exception as e:
            self.display_message(f"An unexpected error occurred while adding movie: {e}", is_error=True)

    def update_movie(self):
        if self.selected_movie_id is None:
            self.display_message("Please select a movie to update.", is_error=True)
            return

        try:
            title = self.update_title_entry.get().strip()
            room = int(self.update_room_entry.get())
            release = self.update_release_entry.get().strip()
            end = self.update_end_entry.get().strip()
            tickets = int(self.update_tickets_entry.get())
            price = float(self.update_price_entry.get())

            if not all([title, release, end]):
                raise ValueError("All update movie fields must be filled.")

            response = self.send_request('update_movie', {
                'id': self.selected_movie_id,
                'title': title,
                'cinema_room': room,
                'release_date': release,
                'end_date': end,
                'tickets_available': tickets,
                'ticket_price': price
            })

            if response['status'] == 'success':
                self.display_message(f"Movie '{title}' updated successfully!", is_error=False)
                self.load_movies()
            else:
                self.display_message(f"Failed to update movie: {response['message']}", is_error=True)
        except ValueError as ve:
            self.display_message(f"Invalid input for update movie: {ve}. Check numbers and dates (YYYY-MM-DD).", is_error=True)
        except Exception as e:
            self.display_message(f"An unexpected error occurred while updating movie: {e}", is_error=True)

    def delete_movie(self):
        if self.selected_movie_id is None:
            self.display_message("Please select a movie to delete.", is_error=True)
            return

        if messagebox.askyesno("Confirm Delete", f"Are you sure you want to delete movie ID {self.selected_movie_id}?"):
            response = self.send_request('delete_movie', {'id': self.selected_movie_id})
            if response['status'] == 'success':
                self.display_message(f"Movie ID {self.selected_movie_id} deleted successfully!", is_error=False)
                self.selected_movie_id = None
                self.display_movie_details({})
                self.load_movies()
            else:
                self.display_message(f"Failed to delete movie: {response['message']}", is_error=True)

    def display_message(self, message, is_error=False):
        if is_error:
            self.status_label.config(text=message, style="Error.TLabel")
        else:
            self.status_label.config(text=message, style="Success.TLabel")

    def create_widgets(self):
        self.master.tk_setPalette(background='#e0f7fa')

        style = ttk.Style()
        style.configure("TFrame", background='#e0f7fa')
        style.configure("TLabel", background='#e0f7fa', font=('Inter', 10))
        style.configure("Error.TLabel", foreground="red")
        style.configure("Success.TLabel", foreground="green")

        style.configure("TButton", font=('Inter', 10, 'bold'), padding=5, relief="raised", borderwidth=2,
                        foreground='white', background='#00796b')
        style.map("TButton", background=[('active', '#004d40')])
        style.configure("TEntry", font=('Inter', 10), fieldbackground='white')
        style.configure("TCombobox", font=('Inter', 10), fieldbackground='white')

        main_frame = ttk.Frame(self.master, padding="20 20 20 20")
        main_frame.pack(fill=tk.BOTH, expand=True)

        selection_frame = ttk.LabelFrame(main_frame, text="Select Movie & Details", padding="10 10 10 10")
        selection_frame.grid(row=0, column=0, columnspan=2, padx=10, pady=10, sticky="ew")
        selection_frame.columnconfigure(1, weight=1)

        ttk.Label(selection_frame, text="Select Movie:").grid(row=0, column=0, padx=5, pady=5, sticky="w")
        self.movie_selection_combobox = ttk.Combobox(selection_frame, state="readonly", width=40, font=('Inter', 10))
        self.movie_selection_combobox.grid(row=0, column=1, padx=5, pady=5, sticky="ew")
        self.movie_selection_combobox.bind("<<ComboboxSelected>>", self.on_movie_selected)

        details_frame = ttk.Frame(selection_frame, padding="5")
        details_frame.grid(row=1, column=0, columnspan=2, padx=5, pady=5, sticky="ew")
        details_frame.columnconfigure(1, weight=1)

        self.detail_title_var = tk.StringVar()
        self.detail_room_var = tk.StringVar()
        self.detail_release_var = tk.StringVar()
        self.detail_end_var = tk.StringVar()
        self.detail_tickets_var = tk.StringVar()
        self.detail_price_var = tk.StringVar()

        ttk.Label(details_frame, textvariable=self.detail_title_var, font=('Inter', 12, 'bold')).grid(row=0, column=0, columnspan=2, sticky="w", pady=2)
        ttk.Label(details_frame, textvariable=self.detail_room_var).grid(row=1, column=0, sticky="w", pady=2)
        ttk.Label(details_frame, textvariable=self.detail_release_var).grid(row=2, column=0, sticky="w", pady=2)
        ttk.Label(details_frame, textvariable=self.detail_end_var).grid(row=3, column=0, sticky="w", pady=2)
        ttk.Label(details_frame, textvariable=self.detail_tickets_var).grid(row=4, column=0, sticky="w", pady=2)
        ttk.Label(details_frame, textvariable=self.detail_price_var).grid(row=5, column=0, sticky="w", pady=2)


        purchase_frame = ttk.LabelFrame(main_frame, text="Purchase Tickets", padding="10 10 10 10")
        purchase_frame.grid(row=1, column=0, padx=10, pady=10, sticky="nsew")
        purchase_frame.columnconfigure(1, weight=1)

        ttk.Label(purchase_frame, text="Customer Name:").grid(row=0, column=0, padx=5, pady=5, sticky="w")
        self.customer_name_entry = ttk.Entry(purchase_frame, width=30, font=('Inter', 10))
        self.customer_name_entry.grid(row=0, column=1, padx=5, pady=5, sticky="ew")

        ttk.Label(purchase_frame, text="Quantity:").grid(row=1, column=0, padx=5, pady=5, sticky="w")
        self.quantity_entry = ttk.Entry(purchase_frame, width=10, font=('Inter', 10))
        self.quantity_entry.grid(row=1, column=1, padx=5, pady=5, sticky="w")
        self.quantity_entry.insert(0, "1")

        self.purchase_button = ttk.Button(purchase_frame, text="Purchase Tickets", command=self.purchase_tickets)
        self.purchase_button.grid(row=2, column=0, columnspan=2, pady=10)

        self.total_cost_label = ttk.Label(purchase_frame, text="Total Cost: R0.00", font=('Inter', 11, 'bold'))
        self.total_cost_label.grid(row=3, column=0, columnspan=2, pady=5)

        add_movie_frame = ttk.LabelFrame(main_frame, text="Add New Movie", padding="10 10 10 10")
        add_movie_frame.grid(row=1, column=1, padx=10, pady=10, sticky="nsew")
        add_movie_frame.columnconfigure(1, weight=1)

        labels = ["Title:", "Room (1-7):", "Release Date (YYYY-MM-DD):", "End Date (YYYY-MM-DD):", "Tickets Available:", "Ticket Price:"]
        self.new_entries = {}
        for i, text in enumerate(labels):
            ttk.Label(add_movie_frame, text=text).grid(row=i, column=0, padx=5, pady=2, sticky="w")
            entry = ttk.Entry(add_movie_frame, width=30, font=('Inter', 10))
            entry.grid(row=i, column=1, padx=5, pady=2, sticky="ew")
            self.new_entries[text.split('(')[0].strip().replace(':', '').replace(' ', '_').lower()] = entry

        self.new_title_entry = self.new_entries['title']
        self.new_room_entry = self.new_entries['room']
        self.new_release_entry = self.new_entries['release_date']
        self.new_end_entry = self.new_entries['end_date']
        self.new_tickets_entry = self.new_entries['tickets_available']
        self.new_price_entry = self.new_entries['ticket_price']

        self.add_movie_button = ttk.Button(add_movie_frame, text="Add Movie", command=self.add_movie)
        self.add_movie_button.grid(row=len(labels), column=0, columnspan=2, pady=10)

        update_delete_frame = ttk.LabelFrame(main_frame, text="Update/Delete Movie", padding="10 10 10 10")
        update_delete_frame.grid(row=2, column=0, columnspan=2, padx=10, pady=10, sticky="ew")
        update_delete_frame.columnconfigure(1, weight=1)

        labels = ["Title:", "Room (1-7):", "Release Date (YYYY-MM-DD):", "End Date (YYYY-MM-DD):", "Tickets Available:", "Ticket Price:"]
        self.update_entries = {}
        for i, text in enumerate(labels):
            ttk.Label(update_delete_frame, text=text).grid(row=i, column=0, padx=5, pady=2, sticky="w")
            entry = ttk.Entry(update_delete_frame, width=30, font=('Inter', 10))
            entry.grid(row=i, column=1, padx=5, pady=2, sticky="ew")
            self.update_entries[text.split('(')[0].strip().replace(':', '').replace(' ', '_').lower()] = entry

        self.update_title_entry = self.update_entries['title']
        self.update_room_entry = self.update_entries['room']
        self.update_release_entry = self.update_entries['release_date']
        self.update_end_entry = self.update_entries['end_date']
        self.update_tickets_entry = self.update_entries['tickets_available']
        self.update_price_entry = self.update_entries['ticket_price']

        button_frame = ttk.Frame(update_delete_frame)
        button_frame.grid(row=len(labels), column=0, columnspan=2, pady=10)

        self.update_movie_button = ttk.Button(button_frame, text="Update Movie", command=self.update_movie)
        self.update_movie_button.pack(side=tk.LEFT, padx=5)

        self.delete_movie_button = ttk.Button(button_frame, text="Delete Movie", command=self.delete_movie)
        self.delete_movie_button.pack(side=tk.LEFT, padx=5)

        self.status_label = ttk.Label(main_frame, text="Ready", font=('Inter', 10, 'italic'), anchor="w", style="Success.TLabel")
        self.status_label.grid(row=3, column=0, columnspan=2, padx=10, pady=5, sticky="ew")


if __name__ == "__main__":
    root = tk.Tk()
    app = CinemaClientApp(root)
    root.mainloop()
    if app.client_socket:
        app.client_socket.close()
